# Greywater AI Project - Exploratory Data Analysis & Data Quality Validation

## 1. Project Objective
The objective of this analysis phase is to conduct rigorous data quality verification, exploratory data analysis (EDA), and architectural profiling on the greywater dataset (`dataset/main.csv`).

This process ensures that:
- Physicochemical and microbiological parameters are thoroughly profiled.
- Data integrity anomalies (missing values, duplicates, out-of-range sensor readings) are identified.
- Source-wise effluent variability (Bathroom, Kitchen, Laundry, Mixed) is quantified.
- The existing target column (`Water_Quality_Class`) is critically evaluated for suitability regarding the planned 4-class smart routing architecture.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Styling configuration
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.size': 10})

# Ensure reports directory exists
os.makedirs('../reports/figures', exist_ok=True)
print("Environment and libraries configured successfully.")


## 2. Dataset Loading
Load the raw dataset from `../dataset/main.csv` using `pandas` and inspect initial records.


In [ ]:
DATASET_PATH = '../dataset/main.csv'

# Load raw dataset without transformations
df = pd.read_csv(DATASET_PATH)

print(f"Dataset successfully loaded from: {DATASET_PATH}")
print(f"Total Observations (Rows): {df.shape[0]}")
print(f"Total Attributes (Columns): {df.shape[1]}")


In [ ]:
# Display the first 5 records
df.head()


In [ ]:
# Display the last 5 records
df.tail()


## 3. Dataset Structure
Verify column names, data storage types, and memory allocation.


In [ ]:
# Column names, non-null counts, and memory footprint
df.info()


## 4. Data Quality Validation
Assess data sanity across numerical and categorical features:
- Missing value counts
- Infinite value counts
- Unexpected negative values for strictly non-negative parameters
- Range plausibility for physical attributes (e.g. pH within 0-14)


In [ ]:
num_cols = [
    'pH', 'TEMP_C', 'SAL_ppt', 'TUR_NTU', 'DS_mg_L', 'TDS_mg_L',
    'TSS_mg_L', 'COND_uS_cm', 'DO_mg_L', 'BOD_mg_L', 'COD_mg_L',
    'NH4F_mg_L', 'NO3_mg_L', 'K_mg_L', 'E_coli_CFU_100mL'
]

# Validation table
quality_records = []
for col in df.columns:
    s = df[col]
    dtype = str(s.dtype)
    missing = int(s.isnull().sum())
    unique = int(s.nunique())
    
    if col in num_cols:
        c_min = round(float(s.min()), 2)
        c_max = round(float(s.max()), 2)
        c_mean = round(float(s.mean()), 2)
        inf_count = int(np.isinf(s).sum())
        neg_count = int((s < 0).sum())
    else:
        c_min, c_max, c_mean = 'N/A', 'N/A', 'N/A'
        inf_count, neg_count = 0, 0
        
    quality_records.append({
        'Column': col,
        'Data Type': dtype,
        'Missing Count': missing,
        'Unique Count': unique,
        'Min': c_min,
        'Max': c_max,
        'Mean': c_mean,
        'Infinite Count': inf_count,
        'Negative Count': neg_count
    })

quality_summary = pd.DataFrame(quality_records)
quality_summary


## 5. Missing Value Analysis
Evaluate null distributions across all features.


In [ ]:
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing_counts, 'Percentage': missing_percent})
print("Missing Value Summary:")
print(missing_df[missing_df['Missing_Count'] > 0] if (missing_df['Missing_Count'] > 0).any() else "No missing values found across any feature (0.00% missingness).")


## 6. Duplicate Analysis
Verify whether identical duplicate records or duplicated `Sample_ID` keys exist.


In [ ]:
total_duplicate_rows = df.duplicated().sum()
sample_id_duplicates = df['Sample_ID'].duplicated().sum()

print(f"Total Identical Duplicate Rows: {total_duplicate_rows}")
print(f"Total Duplicate Sample_ID Keys: {sample_id_duplicates}")


## 7. Feature Type Identification
Organize the 18 columns into functional domain groupings:
1. **Identifier**: Primary database key
2. **Categorical Metadata**: Greywater source origin
3. **Numerical Water Quality Predictors**: Physicochemical & microbiological parameters
4. **Existing Target**: Legacy single-class target column


In [ ]:
identifiers = ['Sample_ID']
categoricals = ['Greywater_Source']
targets = ['Water_Quality_Class']
numerical_features = [c for c in df.columns if c not in identifiers + categoricals + targets]

print("=== Feature Categorization ===")
print(f"A. Identifier Columns ({len(identifiers)}): {identifiers}")
print(f"B. Categorical Metadata ({len(categoricals)}): {categoricals}")
print(f"C. Existing Target ({len(targets)}): {targets}")
print(f"D. Numerical Predictors ({len(numerical_features)}): {numerical_features}")


## 8. Greywater Source Distribution
Quantify the distribution of samples across the four effluent origins: `Laundry`, `Bathroom`, `Mixed`, and `Kitchen`.


In [ ]:
source_counts = df['Greywater_Source'].value_counts()
source_percentages = (df['Greywater_Source'].value_counts(normalize=True) * 100).round(2)
source_summary = pd.DataFrame({'Sample_Count': source_counts, 'Percentage': source_percentages})
print("Greywater Source Summary:")
source_summary


In [ ]:
plt.figure(figsize=(8, 5))
colors = ['#2b5c8f', '#4682b4', '#5f9ea0', '#e07a5f']
ax = sns.barplot(x=source_counts.index, y=source_counts.values, hue=source_counts.index, palette=colors, legend=False)
plt.title('Distribution of Greywater Samples by Source', fontsize=12, weight='bold', pad=12)
plt.xlabel('Greywater Source', fontsize=11, weight='bold')
plt.ylabel('Number of Samples', fontsize=11, weight='bold')

for p in ax.patches:
    pct = (p.get_height() / len(df)) * 100
    ax.annotate(f"{int(p.get_height())}\n({pct:.1f}%)", 
                (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', fontsize=10, color='white', weight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/greywater_source_distribution.png', dpi=300)
plt.close()


## 9. Numerical Feature Statistics
Compute descriptive summary statistics across all 15 numerical parameters and export to `reports/dataset_statistics.csv`.


In [ ]:
stats_list = []
for col in numerical_features:
    s = df[col]
    stats_list.append({
        'Parameter': col,
        'Count': int(s.count()),
        'Mean': round(float(s.mean()), 2),
        'Std': round(float(s.std()), 2),
        'Min': round(float(s.min()), 2),
        '25%': round(float(s.quantile(0.25)), 2),
        'Median': round(float(s.median()), 2),
        '75%': round(float(s.quantile(0.75)), 2),
        'Max': round(float(s.max()), 2)
    })
stats_df = pd.DataFrame(stats_list)
stats_df.to_csv('../reports/dataset_statistics.csv', index=False)
print("Saved summary statistics to reports/dataset_statistics.csv")
stats_df


## 10. Parameter Distribution Analysis
Visualize parameter distributions using histograms with Kernel Density Estimation (KDE) overlays and boxplots.


In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(16, 18))
axes = axes.flatten()

for i, col in enumerate(numerical_features):
    sns.histplot(df[col], kde=True, ax=axes[i], color='#1f77b4', bins=30)
    axes[i].set_title(col, fontsize=11, weight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency')

plt.suptitle('Histograms & KDE of Numerical Water Quality Parameters', fontsize=15, weight='bold', y=1.002)
plt.tight_layout()
plt.savefig('../reports/figures/parameter_distributions_hist.png', dpi=300)
plt.close()


In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(16, 18))
axes = axes.flatten()

for i, col in enumerate(numerical_features):
    sns.boxplot(y=df[col], ax=axes[i], color='#aec7e8')
    axes[i].set_title(f"Boxplot: {col}", fontsize=11, weight='bold')
    axes[i].set_ylabel('')

plt.suptitle('Boxplots for Outlier Detection Across Water Quality Parameters', fontsize=15, weight='bold', y=1.002)
plt.tight_layout()
plt.savefig('../reports/figures/parameter_distributions_box.png', dpi=300)
plt.close()


## 11. Outlier Analysis
Calculate Interquartile Range (IQR) outlier boundaries ($Q1 - 1.5 \times IQR$, $Q3 + 1.5 \times IQR$) for each parameter.
*Note: Extreme values reflect realistic greywater contamination spikes and are retained without deletion.*


In [ ]:
outlier_list = []
for col in numerical_features:
    s = df[col]
    q1 = float(s.quantile(0.25))
    q3 = float(s.quantile(0.75))
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = s[(s < lower) | (s > upper)]
    n_outliers = len(outliers)
    pct_outliers = round((n_outliers / len(s)) * 100, 2)
    outlier_list.append({
        'Parameter': col,
        'Q1': round(q1, 2),
        'Q3': round(q3, 2),
        'IQR': round(iqr, 2),
        'Lower_Bound': round(lower, 2),
        'Upper_Bound': round(upper, 2),
        'Outlier_Count': n_outliers,
        'Outlier_Percentage': pct_outliers
    })

outlier_df = pd.DataFrame(outlier_list)
outlier_df.to_csv('../reports/outlier_analysis.csv', index=False)
print("Saved outlier analysis to reports/outlier_analysis.csv")
outlier_df


## 12. Correlation Analysis
Calculate Pearson correlation coefficients across numerical attributes and generate a correlation heatmap.


In [ ]:
corr_matrix = df[numerical_features].corr().round(3)
corr_matrix.to_csv('../reports/correlation_matrix.csv')
print("Saved correlation matrix to reports/correlation_matrix.csv")

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=1.0, vmin=-1.0, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .8}, annot=True, fmt=".2f", annot_kws={"size": 8})
plt.title('Water Quality Parameters - Pearson Correlation Heatmap', fontsize=14, weight='bold', pad=15)
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=300)
plt.close()


In [ ]:
# Highlight Strongest Pairwise Correlations (|r| >= 0.50)
corr_pairs = []
for i in range(len(numerical_features)):
    for j in range(i+1, len(numerical_features)):
        col1 = numerical_features[i]
        col2 = numerical_features[j]
        r = corr_matrix.loc[col1, col2]
        if abs(r) >= 0.50:
            corr_pairs.append({'Feature_1': col1, 'Feature_2': col2, 'Correlation': r})

top_corrs = pd.DataFrame(corr_pairs).sort_values(by='Correlation', ascending=False)
print("Strongest Pairwise Correlations (|r| >= 0.50):")
top_corrs


## 13. Source-Wise Water Quality Analysis
Evaluate how contaminant levels vary across greywater sources (`Bathroom`, `Kitchen`, `Laundry`, `Mixed`).


In [ ]:
source_quality = df.groupby('Greywater_Source')[numerical_features].mean().round(2).reset_index()
source_quality.to_csv('../reports/source_quality_statistics.csv', index=False)
print("Saved source quality statistics to reports/source_quality_statistics.csv")
source_quality


In [ ]:
focus_params = ['pH', 'TUR_NTU', 'TDS_mg_L', 'TSS_mg_L', 'DO_mg_L', 'BOD_mg_L', 'COD_mg_L', 'E_coli_CFU_100mL']
fig, axes = plt.subplots(4, 2, figsize=(15, 16))
axes = axes.flatten()

for i, col in enumerate(focus_params):
    sns.boxplot(x='Greywater_Source', y=col, data=df, ax=axes[i], hue='Greywater_Source', palette='Set2', legend=False)
    axes[i].set_title(f"Source vs. {col}", fontsize=11, weight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel(col)

plt.suptitle('Source-Wise Comparison of Key Contaminants & Indicators', fontsize=15, weight='bold', y=1.002)
plt.tight_layout()
plt.savefig('../reports/figures/source_wise_comparison.png', dpi=300)
plt.close()


## 14. Existing Target Analysis
Evaluate the existing label column `Water_Quality_Class`.


In [ ]:
target_counts = df['Water_Quality_Class'].value_counts(dropna=False)
target_pct = (df['Water_Quality_Class'].value_counts(normalize=True) * 100).round(2)
target_summary = pd.DataFrame({'Count': target_counts, 'Percentage': target_pct})

print("Existing Target Summary (Water_Quality_Class):")
print(target_summary)


In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.barplot(x=target_counts.index, y=target_counts.values, color='#d95f02')
plt.title('Existing Target Distribution (Water_Quality_Class)', fontsize=12, weight='bold', pad=12)
plt.xlabel('Target Class', fontsize=11, weight='bold')
plt.ylabel('Count', fontsize=11, weight='bold')
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())} (100.0%)", 
                (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', fontsize=11, color='white', weight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/target_distribution.png', dpi=300)
plt.close()


### Critical Findings on Target Suitability
1. **Single-Class Degeneracy**: 100% of samples (1,500/1,500) possess the identical label: `"Needs Treatment"`.
2. **Supervised ML Infeasibility**: A target variable with zero class variance ($y = \text{constant}$) cannot be used to train supervised multi-class classifiers like XGBoost or Random Forest (zero cross-entropy gradient, degenerate accuracy).
3. **Absence of 4 Routing Tiers**: The planned routing categories (`Indoor Reuse`, `Restricted Irrigation`, `Bio-filtration`, `Sewer Bypass`) are completely absent from the raw dataset.
4. **Phase 3 Requirement**: Ground-truth routing targets must be systematically engineered in Phase 3 based on established environmental standards and water quality threshold rules.


## 15. Data Leakage Check
Analyze role and eligibility of each column for downstream machine learning:

1. **`Sample_ID` (Identifier)**:
   - Arbitrary sequential integer index ($1 \dots 1500$).
   - Must be dropped from feature matrix $X$ to prevent spurious memorization and data leakage.

2. **`Greywater_Source` (Categorical Context)**:
   - High operational relevance (Bathroom, Kitchen, Laundry, Mixed indicate distinct contamination signatures).
   - Can be one-hot encoded or label-encoded as an auxiliary feature, or evaluated as a conditioning variable.

3. **`Water_Quality_Class` (Target Label)**:
   - Existing single-class column must NEVER be included as a predictive feature in $X$.
   - Must be replaced by the engineered 4-tier routing target in Phase 3.

4. **Numerical Predictors (15 Physicochemical & Biological Features)**:
   - Legitimate predictors representing real-time sensor measurements.


In [ ]:
# Feature leakage inspection table
leakage_audit = pd.DataFrame([
    {'Column': 'Sample_ID', 'Type': 'Identifier', 'Include in Features (X)?': 'NO', 'Reason': 'Arbitrary sequential ID; causes index memorization.'},
    {'Column': 'Greywater_Source', 'Type': 'Categorical Metadata', 'Include in Features (X)?': 'YES (Encoded)', 'Reason': 'Valid physical metadata indicating effluent source.'},
    {'Column': 'Water_Quality_Class', 'Type': 'Target / Label', 'Include in Features (X)?': 'NO', 'Reason': 'Target label; inclusion would constitute direct target leakage.'},
    {'Column': '15 Numerical Parameters', 'Type': 'Continuous Sensor Metrics', 'Include in Features (X)?': 'YES (Scaled)', 'Reason': 'Primary physical, chemical, and biological sensor telemetry.'}
])
leakage_audit


## 16. Initial Dataset Conclusions
1. **Data Completeness**: Zero missing values across all 18 columns and 1,500 rows. No identical row duplicates or key collisions found.
2. **Plausibility & Integrity**: All values conform to physically plausible ranges (pH 6.0–8.41, Temperatures 18–38°C, non-negative solids, oxygen metrics, and pathogen counts).
3. **Severe Collinearities**: Strong collinearity between `DS_mg_L` and `TDS_mg_L` ($r = 0.994$) and `TUR_NTU` and `TSS_mg_L` ($r = 0.877$). Dimension reduction or feature selection will be addressed in preprocessing.
4. **Source Differences**: Bathroom greywater is the cleanest; Kitchen greywater is severely contaminated with massive organic and bacterial load; Laundry greywater carries high turbidity and surfactant-related conductivity.
5. **Target Gap**: The current target contains solely `"Needs Treatment"`. Systematic multi-tier routing label formulation in Phase 3 is required before any supervised machine learning model training.
